In [16]:
import pandas as pd
import numpy as np
import json
import ast
import re

from pathlib import Path

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Preprocessing environment ready!")

Preprocessing environment ready!


In [17]:
cv_df = pd.read_csv(RAW_DIR / "cv.csv")
job_df = pd.read_csv(RAW_DIR / "job.csv")
matches_df = pd.read_csv(RAW_DIR / "matches.csv")

print("CV:", cv_df.shape)
print("Jobs:", job_df.shape)
print("Matches:", matches_df.shape)

CV: (9544, 11)
Jobs: (1167, 10)
Matches: (122560, 13)


In [18]:
def clean_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [19]:
sample = "  Python Developer \n  with   3 years experience "

print(clean_text(sample))

python developer with 3 years experience


In [20]:
def parse_skills(value):
    if not isinstance(value, str) or not value.strip():
        return []

    try:
        parsed = ast.literal_eval(value)

        if isinstance(parsed, list):
            skills = parsed
        else:
            return []

    except (ValueError, SyntaxError):
        return []

    cleaned = []

    for skill in skills:
        if isinstance(skill, str):
            skill = clean_text(skill)

            if skill and skill not in cleaned:
                cleaned.append(skill)

    return cleaned

In [21]:
test_skills = '["Python", "Machine Learning", "SQL"]'

print(parse_skills(test_skills))

['python', 'machine learning', 'sql']


In [22]:
cv_df["skills_list"] = cv_df["clean_skills_json"].apply(parse_skills)

job_df["skills_list"] = job_df["clean_skills_json"].apply(parse_skills)

display(cv_df[["candidate_id", "clean_skills_json", "skills_list"]].head())

display(job_df[["job_id", "clean_skills_json", "skills_list"]].head())

,candidate_id,clean_skills_json,skills_list
0,cand_00000,"[""big data"", ""hadoop"", ""hive"", ""python"", ""mapr...","[big data, hadoop, hive, python, mapreduce, sp..."
1,cand_00001,"[""data analysis"", ""data analytics"", ""business ...","[data analysis, data analytics, business analy..."
2,cand_00002,"[""software development"", ""machine learning"", ""...","[software development, machine learning, deep ..."
3,cand_00003,"[""accounts payables"", ""accounts receivables"", ...","[accounts payables, accounts receivables, acco..."
4,cand_00004,"[""analytical reasoning"", ""compliance testing k...","[analytical reasoning, compliance testing know..."


,job_id,clean_skills_json,skills_list
0,3651735073,"[""financial management"", ""accounting"", ""gaap p...","[financial management, accounting, gaap princi..."
1,3715891138,"[""recruiting"", ""talent management"", ""complianc...","[recruiting, talent management, compliance, em..."
2,3782812525,"[""sales"", ""customer relationship management"", ...","[sales, customer relationship management, tech..."
3,3861704803,"[""account management"", ""client relationship ma...","[account management, client relationship manag..."
4,3862673730,"[""project management"", ""human resources admini...","[project management, human resources administr..."


In [23]:
print("CV records with empty skill lists:",
      (cv_df["skills_list"].str.len() == 0).sum())

print("Job records with empty skill lists:",
      (job_df["skills_list"].str.len() == 0).sum())

CV records with empty skill lists: 20
Job records with empty skill lists: 0


In [24]:
cv_df["resume_text_clean"] = cv_df["resume_text"].apply(clean_text)

cv_df["education_text_clean"] = cv_df["education_text"].apply(clean_text)

cv_df["experience_text_clean"] = cv_df["experience_text"].apply(clean_text)

job_df["job_description_clean"] = job_df["job_description"].apply(clean_text)

job_df["job_text_clean"] = job_df["job_text"].apply(clean_text)

job_df["education_text_clean"] = job_df["education_text"].apply(clean_text)

In [25]:
display(
    cv_df[["candidate_id", "resume_text_clean"]].head(2)
)

display(
    job_df[["job_id", "job_description_clean"]].head(2)
)

,candidate_id,resume_text_clean
0,cand_00000,target position: senior software engineer obje...
1,cand_00001,target position: machine learning (ml) enginee...


,job_id,job_description_clean
0,3651735073,apply here https://www.indeed.com/job/director...
1,3715891138,human resources manager position summarythe id...


In [26]:
def parse_embedding(value):
    try:
        vector = np.array(
            ast.literal_eval(value),
            dtype=np.float32
        )

        if vector.ndim != 1:
            return None

        if len(vector) != 768:
            return None

        if not np.isfinite(vector).all():
            return None

        return vector

    except (ValueError, SyntaxError, TypeError):
        return None

In [27]:
cv_embeddings = cv_df["embedding"].apply(parse_embedding)

job_embeddings = job_df["embedding"].apply(parse_embedding)

print("Invalid CV embeddings:", cv_embeddings.isna().sum())

print("Invalid Job embeddings:", job_embeddings.isna().sum())

print("Example CV vector shape:", cv_embeddings.iloc[0].shape)

Invalid CV embeddings: 0
Invalid Job embeddings: 0
Example CV vector shape: (768,)


In [28]:
cv_processed = cv_df[
    [
        "candidate_id",
        "target_position",
        "resume_text_clean",
        "skills_list",
        "education_text_clean",
        "experience_text_clean",
        "years_experience",
        "inferred_category"
    ]
].copy()

cv_processed["skills_list"] = cv_processed["skills_list"].apply(
    json.dumps
)

cv_processed.to_csv(
    PROCESSED_DIR / "cv_processed.csv",
    index=False
)

print("Processed CV dataset saved!")
print(cv_processed.shape)

Processed CV dataset saved!
(9544, 8)


In [29]:
job_processed = job_df[
    [
        "job_id",
        "job_title",
        "category",
        "job_description_clean",
        "job_text_clean",
        "skills_list",
        "education_text_clean",
        "years_required"
    ]
].copy()

job_processed["skills_list"] = job_processed["skills_list"].apply(
    json.dumps
)

job_processed.to_csv(
    PROCESSED_DIR / "job_processed.csv",
    index=False
)

print("Processed Job dataset saved!")
print(job_processed.shape)

Processed Job dataset saved!
(1167, 8)


In [30]:
print("CV processed:", cv_processed.shape)
print("Jobs processed:", job_processed.shape)

print("\nCV missing values:")
print(cv_processed.isnull().sum())

print("\nJob missing values:")
print(job_processed.isnull().sum())

print("\nProcessed files:")
print(list(PROCESSED_DIR.iterdir()))

CV processed: (9544, 8)
Jobs processed: (1167, 8)

CV missing values:
candidate_id             0
target_position          0
resume_text_clean        0
skills_list              0
education_text_clean     0
experience_text_clean    0
years_experience         0
inferred_category        0
dtype: int64

Job missing values:
job_id                   0
job_title                0
category                 0
job_description_clean    0
job_text_clean           0
skills_list              0
education_text_clean     0
years_required           0
dtype: int64

Processed files:
[WindowsPath('../data/processed/.gitkeep'), WindowsPath('../data/processed/cv_processed.csv'), WindowsPath('../data/processed/job_processed.csv')]


In [31]:
print("Invalid CV embeddings:", cv_embeddings.isna().sum())

print("Invalid Job embeddings:", job_embeddings.isna().sum())

Invalid CV embeddings: 0
Invalid Job embeddings: 0


In [32]:
print("Empty CV skill lists:",
      (cv_df["skills_list"].str.len() == 0).sum())

print("Empty Job skill lists:",
      (job_df["skills_list"].str.len() == 0).sum())

Empty CV skill lists: 20
Empty Job skill lists: 0


In [33]:
print("Processed CV shape:", cv_processed.shape)

print("Processed Job shape:", job_processed.shape)

Processed CV shape: (9544, 8)
Processed Job shape: (1167, 8)


In [34]:
print("========== CV MISSING VALUES ==========")
print(cv_processed.isnull().sum())

print("\n========== JOB MISSING VALUES ==========")
print(job_processed.isnull().sum())

========== CV MISSING VALUES ==========
candidate_id             0
target_position          0
resume_text_clean        0
skills_list              0
education_text_clean     0
experience_text_clean    0
years_experience         0
inferred_category        0
dtype: int64

========== JOB MISSING VALUES ==========
job_id                   0
job_title                0
category                 0
job_description_clean    0
job_text_clean           0
skills_list              0
education_text_clean     0
years_required           0
dtype: int64


In [36]:
print("=" * 45)
print("SUMMARY")
print("=" * 45)

print("\n1. INVALID EMBEDDINGS")
print("CV:", cv_embeddings.isna().sum())
print("Jobs:", job_embeddings.isna().sum())

print("\n2. EMPTY SKILL LISTS")
print("CV:", (cv_df["skills_list"].str.len() == 0).sum())
print("Jobs:", (job_df["skills_list"].str.len() == 0).sum())

print("\n3. PROCESSED DATASET SHAPES")
print("CV:", cv_processed.shape)
print("Jobs:", job_processed.shape)

print("\n4. MISSING VALUES")
print("CV:", cv_processed.isnull().sum().sum())
print("Jobs:", job_processed.isnull().sum().sum())

print("\n5. PROCESSED FILES")
print(list(PROCESSED_DIR.iterdir()))

print("\n" + "=" * 45)
print("SUMMARY COMPLETED")
print("=" * 45)

SUMMARY

1. INVALID EMBEDDINGS
CV: 0
Jobs: 0

2. EMPTY SKILL LISTS
CV: 20
Jobs: 0

3. PROCESSED DATASET SHAPES
CV: (9544, 8)
Jobs: (1167, 8)

4. MISSING VALUES
CV: 0
Jobs: 0

5. PROCESSED FILES
[WindowsPath('../data/processed/.gitkeep'), WindowsPath('../data/processed/cv_processed.csv'), WindowsPath('../data/processed/job_processed.csv')]

SUMMARY COMPLETED
